In [3]:
import sys
!{sys.executable} -m pip install bdsf astroquery

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 29.4 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.2/112.2 kB 15.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.1 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 10.5 MB/s eta 0:00:00


In [4]:
import os
from pathlib import Path
import itertools

import numpy as np
import bdsf
from astropy.io import fits

from data_tools import fits_to_png, mask_single_image

In [5]:
# -------------------------------
# Core source detection function
# -------------------------------
def detect_source_and_create_mask(
    fits_file: str,
    output_dir: str,
    frequency: float,
    beam: tuple,
    thresh_pix: float,
    thresh_isl: float,
    dilation: int,
    export_png: bool = True,
):
    """
    Run PyBDSF source detection and export mask FITS (and optional PNG).

    Returns
    -------
    mask_path : str
        Path to the generated mask FITS file
    """

    stem = Path(fits_file).stem
    param_tag = f"tp{thresh_pix}_ti{thresh_isl}_d{dilation}"

    mask_dir = Path(output_dir) / stem / "masks"
    masked_png_dir = Path(output_dir) / stem / "masked_png"

    mask_dir.mkdir(parents=True, exist_ok=True)
    masked_png_dir.mkdir(parents=True, exist_ok=True)

    mask_fits_path = mask_dir / f"{stem}_{param_tag}.fits"
    masked_png_path = masked_png_dir / f"{stem}_{param_tag}.png"

    try:
        img = bdsf.process_image(
            fits_file,
            frequency=frequency,
            beam=beam,
            thresh_pix=thresh_pix,
            thresh_isl=thresh_isl,
        )

        img.export_image(
            img_type="island_mask",
            outfile=str(mask_fits_path),
            mask_dilation=dilation,
        )

        if export_png:
            mask_png = fits_to_png(str(mask_fits_path))
            image_png = fits_to_png(fits_file)

            if mask_png is not None and image_png is not None:
                masked_image = mask_single_image(image_png, mask_png)
                masked_image.save(masked_png_path)

        return str(mask_fits_path)

    except Exception as exc:  # pylint: disable=broad-except
        print(f"[ERROR] {fits_file} | {param_tag} | {exc}")

In [6]:
# -------------------------------
# Grid search wrapper
# -------------------------------
def grid_search_source_detection(
    fits_file: str,
    output_dir: str,
    frequency: float,
    beam: tuple,
    threshold_pixel_grid: np.ndarray,
    threshold_island_grid: np.ndarray,
    dilation_grid: np.ndarray,
):
    """
    Perform grid search for a single FITS file.
    """

    for tp, ti, d in itertools.product(
        threshold_pixel_grid,
        threshold_island_grid,
        dilation_grid,
    ):
        detect_source_and_create_mask(
            fits_file=fits_file,
            output_dir=output_dir,
            frequency=frequency,
            beam=beam,
            thresh_pix=tp,
            thresh_isl=ti,
            dilation=d,
        )

In [7]:
# -------------------------------
# Folder-level batch processing
# -------------------------------
def process_fits_folder(
    input_fits_dir: str,
    output_dir: str,
):
    """
    Read all FITS files from a folder and run grid search on each.
    """

    input_fits_dir = Path(input_fits_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    fits_files = sorted(input_fits_dir.glob("*.fits"))

    if not fits_files:
        raise RuntimeError("No FITS files found in input directory.")

    # ---- Grid definition ----
    threshold_pixel = np.array(
        [2.8, 3.2, 3.6, 4.0, 4.4, 4.8, 5.0]
    )
    threshold_island = np.array(
        [1.0, 1.5, 2.0, 2.5, 3.0]
    )
    dilation = np.array([0, 1, 2, 3])

    # FIRST survey defaults
    frequency = 1.4e9
    beam = (0.0005, 0.0005, 0.0)

    for fits_file in fits_files:
        print(f"[INFO] Processing {fits_file.name}")
        grid_search_source_detection(
            fits_file=str(fits_file),
            output_dir=str(output_dir),
            frequency=frequency,
            beam=beam,
            threshold_pixel_grid=threshold_pixel,
            threshold_island_grid=threshold_island,
            dilation_grid=dilation,
        )

In [10]:
# -------------------------------
# CLI entry point
# -------------------------------
if __name__ == "__main__":
    INPUT_FITS_FOLDER = "/home/idies/workspace/Temporary/s.hossain18/scratch/Input_Folder"
    OUTPUT_MASK_FOLDER = "/home/idies/workspace/Temporary/s.hossain18/scratch/Mask_Folder"

    process_fits_folder(
        input_fits_dir=INPUT_FITS_FOLDER,
        output_dir=OUTPUT_MASK_FOLDER,
    )


--> Opened '/home/idies/workspace/Temporary/s.hossain18/scratch/Input_Folder/J065404.9+635115.fits'
Image size .............................. : (300, 300) pixels
Number of channels ...................... : 1
Number of Stokes parameters ............. : 1
Beam shape (major, minor, pos angle) .... : (5.00000e-04, 5.00000e-04, 0.0) degrees


[INFO] Processing J065404.9+635115.fits


Frequency of image ...................... : 1400.000 MHz
Number of blank pixels .................. : 2229 (2.5%)
Flux from sum of (non-blank) pixels ..... : 0.304 Jy
--> Calculating background rms and mean images
Derived rms_box (box size, step size) ... : (35, 12) pixels
--> Variation in rms image significant
--> Using 2D map for background rms
--> Variation in mean image not significant
--> Using constant background mean
Min/max values of background rms map .... : (1.21e-04, 1.76e-04) Jy/beam
Value of background mean ................ : -0.0 Jy/beam
--> Expected 5-sigma-clipped false detection rate < fdr_ratio
--> Using sigma-clipping ('hard') thresholding
Minimum number of pixels per island ..... : 6
Number of islands found ................. : 130


Fitting islands with Gaussians .......... : [|] 0/130

stty: 'standard input': Inappropriate ioctl for device
stty: 'standard input': Inappropriate ioctl for device


...

stty: 'standard input': Inappropriate ioctl for device


-Fitting islands with Gaussians .......... : [/] 1/4\Fitting islands with Gaussians .......... : [-] 2/4Fitting islands with Gaussians .......... : [\] 3/4Fitting islands with Gaussians .......... : [] 4/4[-4G

stty: 'standard input': Inappropriate ioctl for device
stty: 'standard input': Inappropriate ioctl for device
stty: 'standard input': Inappropriate ioctl for device
stty: 'standard input': Inappropriate ioctl for device
Total number of Gaussians fit to image .. : 6
Total flux density in model ............. : 0.423 Jy


stty: 'standard input': Inappropriate ioctl for device
--> Grouping Gaussians into sources
Number of sources formed from Gaussians   : 5


--> Wrote file '/home/idies/workspace/Temporary/s.hossain18/scratch/Mask_Folder/J065419.3+635906/masks/J065419.3+635906_tp5.0_ti3.0_d3.fits'
